# Security-Constrained Unit Commitment (SCUC) - Example 1 (Data from File)
by Xingpeng Li

Converted to Python/Pyomo by Haoxiang Wan

**Lab Website:** [rpglab.github.io](https://rpglab.github.io)

> **Note (M2):** Identical MILP SCUC formulation as `SCUC_IC_e1`.
> Generator and load data are read from `SCUC_IC_e1_M2_data.txt` (AMPL-style `param: SET: col... :=` format).
> Data columns: `gen_min  gen_max  gen_RRlimit  gen_OpCost  gen_NlCost  gen_SuCost` and `Time_TotalPd`.

In [ ]:
from pyomo.environ import *
import re, time

# ─────────────────────────────────────────────────────────────────
# Paths
# ─────────────────────────────────────────────────────────────────
BASE_PATH = '/Users/csab/Desktop/ECE6379_Python_Code/21_SCUC/'
data_file = BASE_PATH + 'SCUC_IC_e1_M2_data.txt'

BigM = 1e3

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Parser  ─  AMPL-style 1D param blocks, inline # comments
# ─────────────────────────────────────────────────────────────────
def parse_ampl_data(filepath):
    with open(filepath, 'r') as f:
        raw = f.readlines()
    lines = [re.sub(r'#.*', '', l).strip() for l in raw]
    lines = [l for l in lines if l]
    text   = ' '.join(lines)
    blocks = [b.strip() for b in re.split(r';', text) if b.strip()]
    sets, params = {}, {}
    for block in blocks:
        m = re.match(r'param\s*:\s*(\w+)\s*:\s*([\w\s]+):=\s*(.*)', block, re.S)
        if not m: continue
        set_name  = m.group(1).strip()
        col_names = m.group(2).split()
        tokens    = m.group(3).split()
        for col in col_names: params.setdefault(col, {})
        it = iter(tokens)
        try:
            while True:
                idx = int(next(it))
                sets.setdefault(set_name, []).append(idx)
                for col in col_names:
                    params[col][idx] = float(next(it))
        except StopIteration:
            pass
    return sets, params


# ─────────────────────────────────────────────────────────────────
# Load data
# ─────────────────────────────────────────────────────────────────
sets, params = parse_ampl_data(data_file)

GEN_data    = sets['GEN']
PERIOD_data = sets['PERIOD']

gen_min_data    = {int(k): v for k, v in params['gen_min'].items()}
gen_max_data    = {int(k): v for k, v in params['gen_max'].items()}
gen_RR_data     = {int(k): v for k, v in params['gen_RRlimit'].items()}
gen_OpCost_data = {int(k): v for k, v in params['gen_OpCost'].items()}
gen_NlCost_data = {int(k): v for k, v in params['gen_NlCost'].items()}
gen_SuCost_data = {int(k): v for k, v in params['gen_SuCost'].items()}
TotalPd_data    = {int(k): v for k, v in params['Time_TotalPd'].items()}

print('GEN    :', GEN_data)
print('PERIOD :', PERIOD_data)
print('TotalPd:', TotalPd_data)
print('gen limits (min/max):', {g: (gen_min_data[g], gen_max_data[g]) for g in GEN_data})
print('SuCost :', gen_SuCost_data, '| NlCost:', gen_NlCost_data)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Build Pyomo Model
# ─────────────────────────────────────────────────────────────────
model = ConcreteModel()

# ── Sets ─────────────────────────────────────────────────────────
model.GEN    = Set(initialize=GEN_data,    ordered=True)
model.PERIOD = Set(initialize=PERIOD_data, ordered=True)

# ── Parameters ───────────────────────────────────────────────────
model.gen_min    = Param(model.GEN,    initialize=gen_min_data)
model.gen_max    = Param(model.GEN,    initialize=gen_max_data)
model.gen_RR     = Param(model.GEN,    initialize=gen_RR_data)
model.gen_OpCost = Param(model.GEN,    initialize=gen_OpCost_data)
model.gen_NlCost = Param(model.GEN,    initialize=gen_NlCost_data)
model.gen_SuCost = Param(model.GEN,    initialize=gen_SuCost_data)
model.TotalPd    = Param(model.PERIOD, initialize=TotalPd_data)

# ── Decision Variables ────────────────────────────────────────────
model.u  = Var(model.GEN, model.PERIOD, within=Binary,           doc='Commitment status')
model.v  = Var(model.GEN, model.PERIOD, within=Binary,           doc='Startup indicator')
model.Pg = Var(model.GEN, model.PERIOD, within=NonNegativeReals, doc='Power output (MW)')

# ── Objective ────────────────────────────────────────────────────
def obj_rule(m):
    return sum(
        m.gen_OpCost[g] * m.Pg[g, t]
        + m.gen_NlCost[g] * m.u[g, t]
        + m.gen_SuCost[g] * m.v[g, t]
        for g in m.GEN for t in m.PERIOD
    )
model.obj = Objective(rule=obj_rule, sense=minimize)

# ── Power balance ─────────────────────────────────────────────────
def PowerBalance_rule(m, t):
    return sum(m.Pg[g, t] for g in m.GEN) == m.TotalPd[t]
model.PowerBalance = Constraint(model.PERIOD, rule=PowerBalance_rule)

# ── Generation limits ─────────────────────────────────────────────
def genLimitMin_rule(m, g, t):
    return m.gen_min[g] * m.u[g, t] <= m.Pg[g, t]
model.genLimitMin = Constraint(model.GEN, model.PERIOD, rule=genLimitMin_rule)

def genLimitMax_rule(m, g, t):
    return m.Pg[g, t] <= m.gen_max[g] * m.u[g, t]
model.genLimitMax = Constraint(model.GEN, model.PERIOD, rule=genLimitMax_rule)

# ── Ramp-up:  Pg[t] - Pg[t-1] <= RR·u[t-1] + BigM·v[t] ─────────
def genRRUp_rule(m, g, t):
    if t == m.PERIOD.first(): return Constraint.Skip
    tp = m.PERIOD.prev(t)
    return m.Pg[g, t] - m.Pg[g, tp] <= m.gen_RR[g] * m.u[g, tp] + BigM * m.v[g, t]
model.genRRUp = Constraint(model.GEN, model.PERIOD, rule=genRRUp_rule)

# ── Ramp-down: Pg[t-1] - Pg[t] <= RR·u[t] + BigM·(v[t]-u[t]+u[t-1]) ─
def genRRDn_rule(m, g, t):
    if t == m.PERIOD.first(): return Constraint.Skip
    tp = m.PERIOD.prev(t)
    return m.Pg[g, tp] - m.Pg[g, t] <= m.gen_RR[g] * m.u[g, t] + BigM * (m.v[g, t] - m.u[g, t] + m.u[g, tp])
model.genRRDn = Constraint(model.GEN, model.PERIOD, rule=genRRDn_rule)

# ── Startup logic: v[t] >= u[t] - u[t-1]  (v[t] >= u[t] at t=1) ─
def genVU_rule(m, g, t):
    if t == m.PERIOD.first():
        return m.v[g, t] >= m.u[g, t]
    tp = m.PERIOD.prev(t)
    return m.v[g, t] >= m.u[g, t] - m.u[g, tp]
model.genVU = Constraint(model.GEN, model.PERIOD, rule=genVU_rule)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Solve
# ─────────────────────────────────────────────────────────────────
solver = SolverFactory('gurobi')
solver.options['mipgap'] = 0.0
solver.options['timelimit'] = 90

start_time = time.time()
results = solver.solve(model, tee=True)
solve_time = time.time() - start_time

# ─────────────────────────────────────────────────────────────────
# Display Results
# ─────────────────────────────────────────────────────────────────
total_cost = sum(
    gen_OpCost_data[g] * model.Pg[g, t].value
    + gen_NlCost_data[g] * model.u[g, t].value
    + gen_SuCost_data[g] * model.v[g, t].value
    for g in model.GEN for t in model.PERIOD
)
print(f'\n=== Optimal Total Cost = ${total_cost:.2f} ===\n')
print(f'  {"Gen":<5} {"Period":<8} {"u":>4} {"v":>4} {"Pg (MW)":>10}')
print('  ' + '-' * 35)
for g in model.GEN:
    for t in model.PERIOD:
        print(f'  G{g:<4} t={t:<6} '
              f'{int(round(model.u[g,t].value)):>4} '
              f'{int(round(model.v[g,t].value)):>4} '
              f'{model.Pg[g,t].value:>10.2f}')
print(f'\nTotal solve elapsed time: {solve_time:.4f} seconds')